In [2]:
import pandas as pd

# قراءة عينة البيانات المنظفة الجاهزة التي نزلتِها من الغيت هاب
df = pd.read_csv(r"C:\Users\dell\ir_project-main\quora_cleaned_sample.csv")
# لعرض أول الأسطر والتأكد أن كل شيء تمام
df.head()

,doc_id,original_text,cleaned_text
0,1,What is the step by step guide to invest in sh...,step step guide invest share market india
1,2,What is the step by step guide to invest in sh...,step step guide invest share market
2,3,What is the story of Kohinoor (Koh-i-Noor) Dia...,story kohinoor diamond
3,4,What would happen if the Indian government sto...,would happen indian government stole kohinoor ...
4,5,How can I increase the speed of my internet co...,increase speed internet connection using vpn


In [3]:
import numpy as np
import pandas as pd

In [4]:
def precision_at_k(retrieved_ids, relevant_ids, k=10):
    # نأخذ أول K وثيقة فقط تم استرجاعها
    top_k_retrieved = retrieved_ids[:k]
    if not top_k_retrieved:
        return 0.0
    
    # حساب عدد الوثائق المشتركة (ذات الصلة الحقيقية)
    true_positives = len(set(top_k_retrieved) & set(relevant_ids))
    return true_positives / len(top_k_retrieved)

In [5]:
# 2. تابع حساب الاستدعاء الكامل (Recall)
# يقيس نسبة الوثائق ذات الصلة التي استطاع النظام استرجاعها من بين جميع الوثائق ذات الصلة المتوفرة
def calculate_recall(retrieved_ids, relevant_ids):
    if not relevant_ids:
        return 0.0
    
    # حساب عدد الوثائق المشتركة المسترجعة
    true_positives = len(set(retrieved_ids) & set(relevant_ids))
    return true_positives / len(relevant_ids)

# --- تجربة برمجية سريعة للتأكد من عمل التوابع ---
mock_retrieved = [1, 5, 12, 4, 9, 20, 22, 14, 3, 11]  # معرفات وثائق رجعها المحرك
mock_relevant = [5, 4, 11, 99, 105]                  # المعرفات الصحيحة من الـ qrel

print(f"Precision@10: {precision_at_k(mock_retrieved, mock_relevant, k=10)}")
print(f"Recall: {calculate_recall(mock_retrieved, mock_relevant)}")

Precision@10: 0.3
Recall: 0.6


In [6]:
# 1. تابع حساب متوسط الدقة (Average Precision - AP) لاستعلام واحد
def calculate_average_precision(retrieved_ids, relevant_ids):
    if not relevant_ids:
        return 0.0
    
    ap_sum = 0.0
    true_positives_count = 0
    
    for rank, doc_id in enumerate(retrieved_ids):
        if doc_id in relevant_ids:
            true_positives_count += 1
            # الدقة عند هذا الترتيب الحالي
            precision_at_rank = true_positives_count / (rank + 1)
            ap_sum += precision_at_rank
            
    if true_positives_count == 0:
        return 0.0
        
    return ap_sum / len(relevant_ids)

In [7]:
# 2. تابع حساب الـ Mean Average Precision (MAP) لجميع الاستعلامات الاختبارية
def calculate_map(all_queries_retrieved, all_queries_relevant):
    # نأخذ متوسط الـ AP لجميع الاستعلامات
    ap_scores = [calculate_average_precision(ret, rel) for ret, rel in zip(all_queries_retrieved, all_queries_relevant)]
    return np.mean(ap_scores) if ap_scores else 0.0

In [8]:
# 3. تابع حساب الـ Normalized Discounted Cumulative Gain (nDCG@K)
def ndcg_at_k(retrieved_ids, relevant_ids, k=10):
    top_k_retrieved = retrieved_ids[:k]
    dcg = 0.0
    
    # حساب DCG (بفرضية التقييم الثنائي: صلة = 1، لا صلة = 0)
    for rank, doc_id in enumerate(top_k_retrieved):
        if doc_id in relevant_ids:
            dcg += 1.0 / np.log2(rank + 2)  # الترتيب يبدأ برقم 1 برمجياً عبر rank + 2
            
    # حساب المثالي Ideal DCG (في حال ظهرت كل العناصر ذات الصلة في البداية)
    idcg = 0.0
    actual_relevant_in_top_k = min(len(relevant_ids), k)
    for rank in range(actual_relevant_in_top_k):
        idcg += 1.0 / np.log2(rank + 2)
        
    if idcg == 0.0:
        return 0.0
        
    return dcg / idcg

In [9]:
# تجميع المقاييس في خدمة مستقلة ونظيفة Clean Architecture
class EvaluationService:
    def __init__(self):
        pass
        
    def evaluate_model(self, all_retrieved, all_relevant, k=10):
        """
        تقوم هذه الدالة بحساب المقاييس الأربعة دفعة واحدة لنموذج معين
        """
        p_at_k_scores = [precision_at_k(ret, rel, k) for ret, rel in zip(all_retrieved, all_relevant)]
        recall_scores = [calculate_recall(ret, rel) for ret, rel in zip(all_retrieved, all_relevant)]
        ndcg_scores = [ndcg_at_k(ret, rel, k) for ret, rel in zip(all_retrieved, all_relevant)]
        map_score = calculate_map(all_retrieved, all_relevant)
        
        return {
            f"Precision@{k}": round(np.mean(p_at_k_scores), 4),
            "Recall": round(np.mean(recall_scores), 4),
            f"nDCG@{k}": round(np.mean(ndcg_scores), 4),
            "MAP": round(map_score, 4)
        }

In [10]:
# --- ميزة الـ RAG الإضافية (Retrieval-Augmented Generation) ---
class RAGService:
    def __init__(self):
        # هنا يتم لاحقاً استدعاء مكتبات مثل HuggingFace أو OpenAI إذا رغبتم
        print("RAG Service Initialized Successfully.")
        
    def generate_smart_answer(self, query, top_retrieved_docs_text):
        """
        تأخذ الاستعلام والوثائق المسترجعة وتولد إجابة ذكية للمستخدم
        """
        # دمج النصوص المسترجعة لتشكل سياق (Context)
        context = " \n ".join(top_retrieved_docs_text[:3]) # نأخذ أول 3 وثائق كأقصى تقدير
        
        # صياغة الـ Prompt للـ LLM
        prompt = f"Based on the following documents:\n{context}\n\nAnswer the user query: {query}"
        
        # كود محاكاة لتوليد النص (سيتم ربطه بـ LLM في مرحلة الربط)
        mock_llm_response = f"[RAG Answer] summary based on the retrieved data for your query: '{query}'."
        return mock_llm_response

In [11]:
# اختبار تكامل الأنظمة (Demo Test)
eval_server = EvaluationService()
rag_server = RAGService()

# بيانات وهمية للاختبار لـ 3 استعلامات
all_retrieved = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
all_relevant = [[2, 10], [5, 6], [1]]

# تشغيل التقييم
results = eval_server.evaluate_model(all_retrieved, all_relevant, k=10)
print("\n=== نتائج تقييم النظام التجريبية ===")
print(results)

# تشغيل الـ RAG التجريبي
sample_docs = ["Python is an interpreted language.", "Information Retrieval is fun."]
rag_response = rag_server.generate_smart_answer("What is Python?", sample_docs)
print("\n=== نتيجة نظام الـ RAG ===")
print(rag_response)

RAG Service Initialized Successfully.

=== نتائج تقييم النظام التجريبية ===
{'Precision@10': np.float64(0.3333), 'Recall': np.float64(0.5), 'nDCG@10': np.float64(0.3601), 'MAP': np.float64(0.2778)}

=== نتيجة نظام الـ RAG ===
[RAG Answer] summary based on the retrieved data for your query: 'What is Python?'.


In [13]:
df.to_csv(r"C:\Users\dell\ir_project-main\quora_cleaned_sample.csv", index=False)